In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Mga input at output ng Primitive

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa page na ito ay binuo gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Ang page na ito ay nagbibigay ng pangkalahatang-ideya ng mga input at output ng mga Qiskit primitive. Sa pamamagitan ng mga primitive na ito maaari kang gumamit ng isang data structure na kilala bilang **Primitive Unified Bloc (PUB)** upang mahusay na tukuyin ang mga vectorized workload. Ang mga PUB na ito ang pangunahing yunit ng trabaho para sa pagpapatupad ng workload. Ginagamit ang mga ito bilang mga input sa `run()` na method para sa mga Sampler at Estimator primitive, na nagpapatupad ng tinukoy na workload bilang isang job. Pagkatapos, kapag nakumpleto na ang job, ang mga resulta ay ibinalik sa isang format na nakasalalay sa mga PUB na ginamit at sa anumang mga tinukoy na opsyon.
<span id="pubs"></span>
## Pangkalahatang-ideya ng mga PUB
Kapag tinatawagan ang pamamaraan ng [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) ng isang primitive, ang pangunahing argument na kailangan ay isang `list` ng isa o higit pang mga tuple — isa para sa bawat circuit na pinapatakbo ng primitive. Ang bawat isa sa mga tuple na ito ay itinuturing na isang PUB, at ang mga kinakailangang elemento ng bawat tuple sa listahan ay nakasalalay sa primitive na ginamit. Ang data na ibinibigay sa mga tuple na ito ay maaari ring ayusin sa iba't ibang hugis upang magbigay ng flexibility sa isang workload sa pamamagitan ng broadcasting — ang mga patakaran nito ay inilarawan sa isang [sumusunod na seksyon](#broadcasting-rules).

### Estimator PUB
Para sa Estimator primitive, ang format ng PUB ay dapat maglaman ng hindi hihigit sa apat na halaga:
- Isang solong `QuantumCircuit`, na maaaring maglaman ng isa o higit pang mga [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) na object
- Isang listahan ng isa o higit pang mga observable, na nagtutukoy ng mga inaasahang halaga na tatantiyahin, na nakaayos sa isang array (halimbawa, isang solong observable na kinakatawan bilang isang 0-d na array, isang listahan ng mga observable bilang isang 1-d na array, at iba pa). Ang data ay maaaring nasa alinman sa `ObservablesArrayLike` na format tulad ng `Pauli`, `SparsePauliOp`, `PauliList`, o `str`.
   > **Note:** Kung mayroon kang dalawang commuting observable sa iba't ibang PUB ngunit may parehong circuit, hindi sila tatantiyahin gamit ang parehong sukat. Ang bawat PUB ay kumakatawan sa ibang batayan para sa pagsukat, kaya naman, kinakailangan ang magkahiwalay na mga pagsukat para sa bawat PUB. Upang matiyak na ang mga commuting observable ay natantiya gamit ang parehong pagsukat, dapat silang ipangkat sa loob ng parehong PUB.
- Isang koleksyon ng mga halaga ng parameter para i-bind ang circuit. Maaari itong tukuyin bilang isang solong array-like na object kung saan ang huling index ay nasa mga `Parameter` na object ng circuit, o maaaring alisin (o katumbas nito, itakda sa `None`) kung ang circuit ay walang mga `Parameter` na object.
- (Opsyonal) isang target na katumpakan para sa mga inaasahang halaga na tatantiyahin

### Sampler PUB
Para sa Sampler primitive, ang format ng PUB tuple ay naglalaman ng hindi hihigit sa tatlong halaga:
- Isang solong `QuantumCircuit`, na maaaring maglaman ng isa o higit pang mga [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) na object
   *Tandaan: Ang mga circuit na ito ay dapat talaga ding maglaman ng mga tagubilin sa pagsukat para sa bawat qubit na susukatin.*
- Isang koleksyon ng mga halaga ng parameter para i-bind ang circuit laban sa $\theta_k$ (kailangan lamang kung may mga `Parameter` na object na ginagamit na dapat i-bind sa runtime)
- (Opsyonal) bilang ng mga shot para sukatin ang circuit
---

Ang sumusunod na code ay nagpapakita ng isang halimbawa ng set ng mga vectorized input sa `Estimator` primitive.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator


import numpy as np

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

<span id="broadcasting"></span>
### Mga patakaran sa broadcasting
Ang mga PUB ay nag-aahon ng mga elemento mula sa maraming array (mga observable at mga halaga ng parameter) sa pamamagitan ng pagsunod sa parehong mga patakaran sa broadcasting ng NumPy. Ang seksyong ito ay maikling buod ng mga patakarang iyon. Para sa detalyadong paliwanag, tingnan ang [dokumentasyon ng broadcasting rules ng NumPy](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Mga patakaran:

* Ang mga input array ay hindi kailangang magkaroon ng parehong bilang ng mga dimensyon.
  * Ang resulting array ay magkakaroon ng parehong bilang ng mga dimensyon katulad ng input array na may pinakamalaking dimensyon.
  * Ang laki ng bawat dimensyon ay ang pinakamalaking laki ng katumbas na dimensyon.
  * Ang mga nawawalang dimensyon ay ipinapalagay na may laki na isa.
* Ang mga paghahambing ng hugis ay nagsisimula sa pinaka-kanang dimensyon at nagpapatuloy sa kaliwa.
* Ang dalawang dimensyon ay tugma kung ang kanilang mga laki ay pantay o kung isa sa kanila ay 1.

Mga halimbawa ng mga pares ng array na nag-broadcast:

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

Mga halimbawa ng mga pares ng array na hindi nag-broadcast:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))

And this DataBin has attributes: dict_keys(['evs', 'stds'])
Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). 

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680

Ang `Estimator` ay nagbabalik ng isang tinatantiyahang inaasahang halaga para sa bawat elemento ng broadcasted shape.

Narito ang ilang halimbawa ng mga karaniwang pattern na ipinahayag sa mga termino ng array broadcasting. Ang kanilang kasamang visual na representasyon ay ipinapakita sa sumusunod na figure:

Ang mga set ng halaga ng parameter ay kinakatawan ng n x m na mga array, at ang mga array ng observable ay kinakatawan ng isa o higit pang single-column na mga array. Para sa bawat halimbawa sa nakaraang code, ang mga set ng halaga ng parameter ay pinagsama sa kanilang observable array upang lumikha ng mga resulting na tinatantiyahang inaasahang halaga.

 - *Halimbawa 1*: (broadcast single observable) ay may set ng halaga ng parameter na isang 5x1 na array at isang 1x1 na observables array. Ang isang item sa observables array ay pinagsama sa bawat item sa set ng halaga ng parameter upang lumikha ng isang solong 5x1 na array kung saan ang bawat item ay isang kombinasyon ng orihinal na item sa set ng halaga ng parameter kasama ang item sa observables array.

 - *Halimbawa 2*: (zip) ay may 5x1 na set ng halaga ng parameter at isang 5x1 na observables array. Ang output ay isang 5x1 na array kung saan ang bawat item ay isang kombinasyon ng ika-n na item sa set ng halaga ng parameter kasama ang ika-n na item sa observables array.

  - *Halimbawa 3*: (outer/product) ay may 1x6 na set ng halaga ng parameter at isang 4x1 na observables array. Ang kanilang kombinasyon ay nagbubunga ng isang 4x6 na array na nilikha sa pamamagitan ng pagsasama ng bawat item sa set ng halaga ng parameter sa *bawat* item sa observables array, kaya ang bawat halaga ng parameter ay nagiging isang buong column sa output.

  - *Halimbawa 4*: (Standard nd generalization) ay may 3x6 na array ng set ng halaga ng parameter at dalawang 3x1 na observables array. Pinagsama ang mga ito upang lumikha ng dalawang 3x6 na output array sa paraang katulad ng nakaraang halimbawa.

![Ang imahe na ito ay nagpapakita ng ilang visual na representasyon ng array broadcasting.](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual na representasyon ng broadcasting")

In [4]:
from qiskit.primitives import StatevectorSampler

# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

sampler = StatevectorSampler()

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=1024, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=1024, num_bits=10>)

The shape of register `meas` is (1024, 2).

The bytes in register `alpha`, shot by shot:
[[  0   0]
 [  3 255]
 [  0   0]
 ...
 [  3 255]
 [  3 255]
 [  3 255]]



:::tip[SparsePauliOp]

Ang bawat `SparsePauliOp` ay binibilang bilang isang solong elemento sa kontekstong ito, anuman ang bilang ng mga Pauli na nasa `SparsePauliOp`. Kaya, para sa layunin ng mga patakarang ito sa broadcasting, lahat ng sumusunod na elemento ay may parehong hugis:

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'0000000000': 492, '1111111111': 532}


Ang mga sumusunod na listahan ng mga operator, habang katumbas sa mga tuntunin ng impormasyong nilalaman, ay may iba't ibang hugis:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results

job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=1024, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=1024, num_bits=9>)


:::

## Pangkalahatang-ideya ng mga output ng primitive
Kapag naipadala na ang isa o higit pang PUB sa isang QPU para sa pagpapatupad at matagumpay na natapos ang isang job, ang data ay ibinabalik bilang isang [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) na container object. Ang `PrimitiveResult` ay naglalaman ng isang iterable na listahan ng mga [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) na object na naglalaman ng mga resulta ng pagpapatupad para sa bawat PUB. Halimbawa, ang isang job na isinumite nang may 20 PUB ay magbabalik ng isang `PrimitiveResult` na object na naglalaman ng listahan ng 20 `PubResult`, isa para sa bawat PUB.

Ang bawat isa sa mga `PubResult` na object na ito ay may parehong `data` at isang opsyonal na `metadata` na attribute. Ang `data` attribute ay isang customized na [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) na naglalaman ng mga tinatantiyahang inaasahang halaga sa kaso ng Estimator, o mga sample ng output ng circuit sa kaso ng Sampler.

Ang `data` attribute ay maaari ring maglaman ng iba pang impormasyon na tukoy sa implementasyon tulad ng mga standard deviation. Ang `metadata` attribute ay maaaring maglaman ng karagdagang impormasyon na tukoy sa implementasyon tungkol sa pagpapatupad ng kaugnay na PUB.

Ang sumusunod ay isang visual na balangkas ng istruktura ng data ng `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Estimator output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object,
        |       |     ## which includes data such as the following:
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    > **Note:** Ang nasa itaas ay isang halimbawa ng data na maaaring ibalik. Ang aktwal na data na ibinalik ay nakasalalay sa implementasyon.
    </TabItem>
    <TabItem value="sampler" label="Sampler output">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data for first PUB (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second PUB
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

### Output ng Estimator
Tulad ng nabanggit kanina, ang data na ibinalik sa `PubResult` para sa Estimator primitive ay nakasalalay sa implementasyon. Halimbawa, maaari itong maglaman ng isang array ng mga expectation value (`PubResult.data.evs`) at mga kaugnay na standard deviation (`PubResult.data.stds`).

Ang code snippet sa ibaba ay nagpapaliwanag ng format ng `PrimitiveResult` (at kaugnay na `PubResult`) para sa job na ginawa sa itaas.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (1024, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (1024, 2).
The bytes in register `beta`, shot by shot:
[[  1 255]
 [  1 255]
 [  1 255]
 ...
 [  0   0]
 [  0   0]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (1024, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  1 255]
 [  1 255]
 [  1 255]
 [  1 255]
 [  1 255]]



Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: -0.017578125
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: -0.017578125

The shape of the merged results is (1024, 2).
The bytes of the merged results:
[[  3 255]
 [  3 255]
 [  3 255]
 ...
 [  0   0]
 [  0   0]
 [  3 255]]



## Result metadata

In addition to the execution results, the `PrimitiveResult` and `PubResult` objects contain an optional metadata attribute about the job that was submitted. The metadata returned (if any) is implementation-specific.

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'version' : 2,

The metadata of the PubResult result is:
'shots' : 1024,
'circuit_metadata' : {},


### Output ng Sampler
Kapag matagumpay na natapos ang isang Sampler job, ang ibinalik na [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) na object ay naglalaman ng listahan ng mga [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult), isa bawat PUB. Ang mga data bin ng mga `SamplerPubResult` na object na ito ay mga dict-like na object na naglalaman ng isang `BitArray` bawat `ClassicalRegister` sa circuit.

Ang klase ng `BitArray` ay isang container para sa ordered na shot data. Sa mas detalyadong pagpapaliwanag, iniiimbak nito ang mga na-sample na bitstring bilang bytes sa loob ng isang two-dimensional array. Ang kaliwang pinaka-axis ng array na ito ay tumatakbo sa mga ordered na shot, habang ang kanang pinaka-axis ay tumatakbo sa mga byte.

Bilang unang halimbawa, tingnan natin ang sumusunod na sampung qubit na circuit: